# Study 879 — Weekly Economic Index 📅

**Does a *weekly* growth nowcast time the market better than the monthly macro tape?**

The **Weekly Economic Index** (Lewis, Mertens & Stock, 2020) blends **ten weekly** activity
series — Redbook retail, jobless claims, tax withholding, rail traffic, fuel sales,
temp-staffing, steel, electricity, consumer confidence — into one real-time nowcast of U.S.
growth, published every week by the **Dallas Fed**. The claim: its **level** and its
**weekly change** should predict **forward SPY** and the **cyclical-vs-defensive rotation**
(consumer-discretionary `XLY` vs consumer-staples `XLP`). We test it on the real workbook
history (2008-01-12 → 2026-06-13, 962 aligned weeks).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `94e9e76c22ef`);
the live cells run the fast synthetic control. The level uses the revised WEI vintage —
magnitudes are an upper bound.*


## 1. The idea in one picture

Monthly data arrives weeks late. A **weekly** nowcast that reads jobless claims, retail sales, rail traffic and eight other series *as they land* should, in principle, let you tilt into cyclicals (`XLY`) and out of defensives (`XLP`) — or just into stocks — *before* the monthly numbers confirm the turn. Higher frequency, more timely signal, better timing. That's the theory.

In [1]:
R = dict(spy1_lvl_t=-1.12, spy1_dwei_t=1.34, rot4_lvl_t=-2.24, spy1_r2=0.0033)
print('forward SPY (1wk) on WEI level : NW t = %+.2f  (wrong sign, insignificant)' % R['spy1_lvl_t'])
print('forward SPY (1wk) on weekly chg: NW t = %+.2f  (right sign, insignificant)' % R['spy1_dwei_t'])
print('XLY-XLP (4wk)   on WEI level  : NW t = %+.2f  (SIGNIFICANT but WRONG sign)' % R['rot4_lvl_t'])
print('regression R^2 (SPY 1wk)      : %.4f  (~0.3%% of forward variance)' % R['spy1_r2'])

forward SPY (1wk) on WEI level : NW t = -1.12  (wrong sign, insignificant)
forward SPY (1wk) on weekly chg: NW t = +1.34  (right sign, insignificant)
XLY-XLP (4wk)   on WEI level  : NW t = -2.24  (SIGNIFICANT but WRONG sign)
regression R^2 (SPY 1wk)      : 0.0033  (~0.3% of forward variance)


## 2. Is the machinery honest? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`) and check the regression recovers it — and that it stays *silent* on the null (`edge=0`, a nowcast that varies but predicts nothing). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from wei import data, strategy as st
null = st.synthetic_detect(data.synthetic(edge=0.0, seed=879, n=700))
planted = st.synthetic_detect(data.synthetic(edge=0.010, seed=879, n=700))
print('null world   : SPY level t = %+.2f  (should be ~0)' % null['t_level'])
print('planted world: SPY level t = %+.2f  (should light up)' % planted['t_level'])

null world   : SPY level t = -0.72  (should be ~0)
planted world: SPY level t = +16.69  (should light up)


## 3. The honest verdict — the nowcast does *not* time the market

On the real tape the WEI **level** does not predict forward SPY (NW *t* = **-1.12**, and the *wrong* sign), and the **weekly change** — the signal that *should* work — is the right sign but insignificant (*t* = **+1.34**). Its one significant hit (the weekly change predicting SPY at *t* = +2.26) lives **entirely in the 2008–09 recession/recovery** and vanishes after 2017 (*t* = +0.24). The only overall \|t\| ≥ 2 slope — the rotation level at 4 weeks (**-2.24**) — is *wrong-signed*: strong growth predicts cyclical *under*-performance (a mean-reversion), the opposite of the claim. **Signal: None**, **Tradability: Mirage** (no overlay beats buy-and-hold). The weekly nowcast is a smooth proxy for the growth cycle, not a market-timing edge.